# S6_LOC_cmv_predict.ipynb
##  Convert all NAT-Files to GeoTiff as multi-threaded Process
## Warp and Clip to Region Western Austria or Vorarlber
Conda Env: scikit-learn
last edit: 13.09.2026
Requirements: Windows 7-zip Installation (64bit)

Copyright (c) 2026 Andreas Mätzler

All rights reserved.

You may not use, copy, modify, distribute, or reproduce this code for any purpose without explicit written permission from the author.

In [1]:
# import necessary python packages
### import py7zr
import os
import gzip
import subprocess
import shutil
import numpy as np
from osgeo import gdal, ogr, osr
import pyresample as pr
from satpy import Scene
import datetime
from datetime import timedelta
from multiprocessing.dummy import Pool as ThreadPool
import time
import pandas as pd

In [2]:
# template from https://www.dariusgoergen.com/contents/blog/2020-06-14-nat2tif/
def wrapper_nat2geotiff(eumetsat_native_output_path, eumetsat_geotiff_timestamped_path, eumetsat_archive_path, filename):
    print("# Process File:  " + filename)

    # check if File is a GZ or 7Z-File
    if (filename.endswith(".gz")) or filename.endswith(".7z") or filename.endswith(".zip") or filename.endswith(".bz2"):
        # get plain Filename without extension
        if (filename.endswith(".gz")) or filename.endswith(".7z"):
            plain_filename= os.path.basename(filename)[:-7]
        else:
            plain_filename= os.path.basename(filename)[:-8]

        # Create rounded Timestamp (to Quarter before) as result TIF-filename
        rounded_timestamp_filename = round_filename_to_quarter(filename = os.path.basename(filename)[0:74])

        # declare area variables
        areas = ["Vorarlberg","Lake of Constance"]

        # loop trought areas for clipped geotiff export
        for area in areas:        
            # create some information on the reference system
            area_id = "Austria West"
            description = "Geographical Coordinate System clipped on Western Austria Region"
            proj_id = "Austria"
            # specifing some parameters of the projection
            proj_dict = {"proj": "longlat", "ellps": "WGS84", "datum": "WGS84"}

            if area == "Vorarlberg":
                # calculate the width and height of the aoi in pixels - region Vorarlberg
                llx = 9 # lower left x coordinate in degrees
                lly = 46 # lower left y coordinate in degrees
                urx = 11 # upper right x coordinate in degrees
                ury = 48 # upper right y coordinate in degrees
                extend='Clip_Vorarlberg'
            else:
                # calculate the width and height of the aoi in pixels - lake of constance
                llx = 7 # lower left x coordinate in degrees
                lly = 45 # lower left y coordinate in degrees
                urx = 13 # upper right x coordinate in degrees
                ury = 50 # upper right y coordinate in degrees
                extend='Clip_Lake_of_Constance'

            # resolution = 0.005 # target resolution in degrees
            resolution = 0.01 # target resolution in degrees
            # calculating the number of pixels
            width = int((urx - llx) / resolution)
            height = int((ury - lly) / resolution)
            area_extent = (llx,lly,urx,ury)
            # defining the area
            area_def = pr.geometry.AreaDefinition(area_id, proj_id, description, proj_dict, width, height, area_extent)
            print(area_def)

            datasets = ['IR_VIS_WV','HRV'] # for production use all channels
            # datasets = ['HRV'] # only HRV-Channel for testing
            # datasets = ['IR_VIS_WV'] # only IR_VIS_WV-Channel for testing

            for dataset in datasets:
                # set filenames as variables 
                archive_filename = os.path.join(eumetsat_archive_path, filename)
                geotiff_filename = os.path.join(eumetsat_geotiff_timestamped_path, extend, rounded_timestamp_filename + "_{}.tif".format(dataset))
                native_filename = os.path.join(eumetsat_native_output_path, plain_filename + ".nat")
                
                # GeoTIFF-File exists?
                if not os.path.isfile(geotiff_filename):
                    print("    GeoTIFF-File {} not exists!".format(geotiff_filename))
                    # NAT-File exists?
                    if not os.path.isfile(native_filename):
                            print("    Extract Native-File {} !".format(native_filename))
                            # use of 7z.exe for performance purpose
                            unzip_command = ['C:\\Program Files\\7-Zip\\7z.exe', 'e', "-o"+ eumetsat_native_output_path, archive_filename, '-y']
                            subprocess.call(unzip_command)

                    # Convert NAT-File to GeoTIFF-Format
                    reader = "seviri_l1b_native"
                    print("    Convert NAT-File to GeoTIFF: {}".format(geotiff_filename))
                    try:
                        nat2tif(file = native_filename, calibration = "radiance", area_def = area_def, dataset = dataset, \
                            reader = reader, outdir = os.path.join(eumetsat_geotiff_timestamped_path, extend), label = dataset, \
                            dtype = "float32", radius = 16000, epsilon = 0.5, nodata = -3.4E+38, outfile = geotiff_filename)
                        print("    Successfully created GeoTIFF: {}".format(geotiff_filename))
                    except Exception as e:
                        print("    ERROR converting {}: {}".format(dataset, str(e)))
                        print("    Skipping dataset {} for this file".format(dataset))
        
        # delete temporary uncompressed NAT-File
        print("### NAT-File: {}".format(native_filename))
        if os.path.exists(native_filename):
            try:
                print("    Deleting NAT-File: {}".format(native_filename))
                os.remove(native_filename)
            except:
                print("    Error: NAT-File {} is corrupted or locked! Skipping...".format(native_filename))
                shutil.copyfile(native_filename, os.path.join(eumetsat_archive_path + "\\..\\defekt", plain_filename + ".nat"))
                os.remove(native_filename)
                pass

In [3]:
def nat2tif(file, calibration, area_def, dataset, reader, outdir, label, dtype, radius, epsilon, nodata, outfile):
  # open the file
  scn = Scene(filenames = {reader: [file]})
 
  # set bands for each dataset
  if dataset == 'HRV':
    bands = ['HRV']
  else:
    # wrong order of bands - alphabetical
    #bands = ['IR_016','IR_039','IR_087','IR_097','IR_108','IR_120','IR_134','VIS006','VIS008','WV_062','WV_073']
    # correct order of bands
    # source: https://eumetsat.int/0-degree-service
    bands = ['VIS006','VIS008','IR_016','IR_039','WV_062','WV_073','IR_087','IR_097','IR_108','IR_120','IR_134']
    
  # set starting band for iteration
  bandnr = 1

  for band in bands:
     # let us check that the specified data set is actually available
    scn_names = scn.all_dataset_names()
    # raise exception if dataset is not present in available names
    if band not in scn_names:
      raise Exception("Specified dataset is not available.")
    
    # output band name
    print("       Execute Band {} as Bandnr.{}".format(band,bandnr))
    # we need to load the data, different calibration can be chosen
    scn.load([band], calibration=calibration)
    # let us extract the longitude and latitude data
    lons, lats = scn[band].area.get_lonlats()
    # now we can apply a swath definition for our output raster
    swath_def = pr.geometry.SwathDefinition(lons=lons, lats=lats)
    # and finally we also extract the data
    values = scn[band].values
    # we will now change the datatype of the arrays
    # depending on the present data this can be changed
    lons = lons.astype(dtype)
    lats = lats.astype(dtype)
    values = values.astype(dtype)

    # now we can already resample our data to the area of interest
    values = pr.kd_tree.resample_nearest(swath_def, values,
                                              area_def,
                                              radius_of_influence=radius, # in meters
                                              epsilon=epsilon,
                                              fill_value=False)
    # we are going to check if the outdir exists and create it if it doesnt
    
    print("       Band values:",band, np.min(values),np.max(values))
    
    if not os.path.exists(outdir):
      os.makedirs(outdir)
    
    # now we define some metadata for our raster file
    cols = values.shape[1]
    rows = values.shape[0]
    pixelWidth = (area_def.area_extent[2] - area_def.area_extent[0]) / cols
    pixelHeight = (area_def.area_extent[1] - area_def.area_extent[3]) / rows
    originX = area_def.area_extent[0]
    originY = area_def.area_extent[3] 
    # create output is just for the first band necessary
    if bandnr == 1:
        # you can change the dataformat but be sure to be able to store negative values including -9999
        dst_datatype = gdal.GDT_Float32
        # here we actually create the file
        driver = gdal.GetDriverByName("GTiff")
        # GeoTIFF Options from https://kokoalberti.com/articles/geotiff-compression-optimization-guide/
        outRaster = driver.Create(outfile, cols, rows, len(bands), dst_datatype, [ 'COMPRESS=ZSTD', 'PREDICTOR=3', 'TILED=YES', 'NUM_THREADS=ALL_CPUS' ] )
        # writing the metadata
        outRaster.SetGeoTransform((originX, pixelWidth, 0, originY, 0, pixelHeight))
    # set band name to geotiff band description
    outRaster.GetRasterBand(bandnr).SetDescription(band)
    # creating a new band and writting the data
    outband = outRaster.GetRasterBand(bandnr)
    outband.WriteArray(np.array(values)) # writting the values
    outband.SetNoDataValue(nodata) #specified no data value by user
    outRasterSRS = osr.SpatialReference() # create CRS instance
    outRasterSRS.ImportFromEPSG(4326) # get info for EPSG 4326
    outRaster.SetProjection(outRasterSRS.ExportToWkt()) # set CRS as WKT
    # increase the bandnr
    bandnr = bandnr + 1
  # clean up
  outband = None
  outRaster.FlushCache()
  outRaster = None
  del file,  scn, outband, outRaster

In [4]:
def round_filename_to_quarter(filename):
    date_object = datetime.datetime.strptime(filename[24:38], '%Y%m%d%H%M%S')
    rounded = date_object - (date_object - date_object.min) % timedelta(minutes=15)
    print("    File-Timestamp:    " + date_object.strftime("%Y-%m-%d %H_%M_%S"))  # printed in default formatting
    rounded_str=rounded.strftime("%Y-%m-%d %H_%M_%S")
    print("    Rounded-Timestamp: " + rounded_str)  # printed in default formatting
    return rounded_str

In [5]:
if __name__ == '__main__':
    from concurrent.futures import ThreadPoolExecutor
    from concurrent.futures import as_completed
    from itertools import repeat
    
    # set variables
    eumetsat_path = "C:\\Users\\Andreas\\Documents\\UNIGIS\\2017\\Master-Thesis\\Daten\\Satellite\\EUMETSAT"
    eumetsat_native_output_path = eumetsat_path + "\\Native"
    eumetsat_geotiff_timestamped_path = eumetsat_path + "\\Result_Timestamped\\GeoTIFF\\TEST"
    eumetsat_archive_path = "F:\\EUMETSAT\\TEST"
    
    # create output folder
    if not os.path.exists(eumetsat_native_output_path):
         os.makedirs(eumetsat_native_output_path)
    if not os.path.exists(eumetsat_geotiff_timestamped_path):
         os.makedirs(eumetsat_geotiff_timestamped_path)
    
    # create empty list for files
    files = []
        
    # Loop all compressed NAT-Files
    for file in os.listdir(eumetsat_archive_path):
        files.append(file)
    
    # create multiple Threads for parallel processing
    with ThreadPoolExecutor(max_workers = 8) as executor:
        results = executor.map(wrapper_nat2geotiff, repeat(eumetsat_native_output_path), repeat(eumetsat_geotiff_timestamped_path), repeat(eumetsat_archive_path), files)
    for result in results:
        print(result)

# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231121243.496000000Z-20181231121300-1404462-6.nat.gz
    File-Timestamp:    2018-12-31 12_12_43
    Rounded-Timestamp: 2018-12-31 12_00_00
# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231122743.241000000Z-20181231122800-1404462-6.nat.gz
    File-Timestamp:    2018-12-31 12_27_43
    Rounded-Timestamp: 2018-12-31 12_15_00
Area ID: Austria West
Description: Austria
Projection ID: Geographical Coordinate System clipped on Western Austria Region
Projection: {'datum': 'WGS84', 'no_defs': 'None', 'proj': 'longlat', 'type': 'crs'}
Number of columns: 200
Number of rows: 200
Area extent: (9, 46, 11, 48)
Area ID: Austria West
Description: Austria
Projection ID: Geographical Coordinate System clipped on Western Austria Region
Projection: {'datum': 'WGS84', 'no_defs': 'None', 'proj': 'longlat', 'type': 'crs'}
Number of columns: 200
Number of rows: 200
Area extent: (9, 46, 11, 48)
# Process File:  MSG4-SEVI-MSG15-0100-NA-20181231124242.986000000Z-20181